In [0]:
passengers_day1 = [
(101,"Rahul Sharma","Hyderabad","Economy","India"),
(102,"Priya Reddy","Bangalore","Business","India"),
(103,"Amit Kumar","Mumbai","Economy","India"),
(104,"Sneha Patel","Delhi","Premium Economy","India"),
(105,"Farhan Ali","Chennai","Economy","India")
]

columns = [
"passenger_id",
"passenger_name",
"city",
"travel_class",
"country"
]

df_day1 = spark.createDataFrame(
    passengers_day1,
    columns
)

df_day1.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
passengers_day2 = [
(102,"Priya Reddy","Bangalore","First Class","India"),
(104,"Sneha Patel","Hyderabad","Premium Economy","India"),
(106,"Neha Singh","Pune","Economy","India"),
(107,"Arjun Verma","Kochi","Business","India")
]

df_day2 = spark.createDataFrame(
    passengers_day2,
    columns
)

df_day2.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
# Location inside the Unity Catalog volume
volume_path = "/Volumes/hexa_databricks_7405610905139839/default/hexawarelake13/deltabrick"

# Write Delta files
(
    df_day1.write
    .format("delta")
    .mode("overwrite")
    .save(volume_path)
)

# Read Delta files
delta_df = (
    spark.read
    .format("delta")
    .load(volume_path)
)


display(delta_df)

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Business,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Delhi,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India


In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(
    spark,
    "/Volumes/hexa_databricks_7405610905139839/default/hexawarelake13/deltabrick"
)
deltaTable.history().show(truncate=False)

+-------+-------------------+---------------+-------------------------------------------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
deltaTable.alias("target").merge(
    df_day2.alias("source"),
    "target.passenger_id = source.passenger_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
df_day1.write.format("delta").mode("overwrite") \
    .saveAsTable("passengers_delta")

In [0]:
%sql
select * from passengers_delta

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Business,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Delhi,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark,
    "passengers_delta"
)

delta_table.alias("target").merge(
    df_day2.alias("source"),
    "target.passenger_id = source.passenger_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from passengers_delta

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
103,Amit Kumar,Mumbai,Economy,India
105,Farhan Ali,Chennai,Economy,India
104,Sneha Patel,Hyderabad,Premium Economy,India
102,Priya Reddy,Bangalore,First Class,India
107,Arjun Verma,Kochi,Business,India
106,Neha Singh,Pune,Economy,India


In [0]:
%sql
SELECT *
FROM default.passengers_delta
ORDER BY passenger_id;

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Premium Economy,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Hyderabad,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India
106,Neha Singh,Pune,Economy,India
107,Arjun Verma,Kochi,Business,India
131,John,New York,First Class,USA


In [0]:
print('day 01 count: ', df_day1.count())
print('day 02 count: ', df_day2.count())
print("Record Count:", spark.table("passengers_delta").count())

day 01 count:  5
day 02 count:  4
Record Count: 7


In [0]:
%sql DESCRIBE HISTORY passengers_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-06-18T10:15:08.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3991796108455833),9f294236-827d-414c-80db-af47f0e91076,0618-092705-x2bw3462-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 9131, p25FileSize -> 2048, numDeletionVectorsRemoved -> 1, minFileSize -> 2048, numAddedFiles -> 1, maxFileSize -> 2048, p75FileSize -> 2048, p50FileSize -> 2048, numAddedBytes -> 2048)",null,Databricks-Runtime/18.2.x-photon-scala2.13
4,2026-06-18T10:15:05.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(passenger_id#17756L = passenger_id#17781L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3991796108455833),9f294236-827d-414c-80db-af47f0e91076,0618-092705-x2bw3462-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 7135, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3758, materializeSourceTimeMs -> 131, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1376, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2208)",null,Databricks-Runtime/18.2.x-photon-scala2.13
3,2026-06-18T10:14:50.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3991796108455833),4cf770d8-b002-4950-9e09-c085024a1d62,0618-092705-x2bw3462-v2n,2,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 2048, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1996)",null,Databricks-Runtime/18.2.x-photon-scala2.13
2,2026-06-18T10:10:47.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3991796108455833),86d83a98-7f53-4c0f-b379-4716f96513c8,0618-092705-x2bw3462-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 9131, p25FileSize -> 2048, numDeletionVectorsRemoved -> 1, minFileSize -> 2048, numAddedFiles -> 1, maxFileSize -> 2048, p75FileSize -> 2048, p50FileSize -> 2048, numAddedBytes -> 2048)",null,Databricks-Runtime/18.2.x-photon-scala2.13
1,2026-06-18T10:10:46.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(passenger_id#15750L = passenger_id#15775L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3991796108455833),86d83a98-7f53-4c0f-b379-4716f96513c8,0618-092705-x2bw3462-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 7135, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRo

In [0]:
%sql
SELECT passenger_id,
       passenger_name,
       city,
       travel_class
FROM passengers_delta VERSION AS OF 0
WHERE passenger_id = 102;

passenger_id,passenger_name,city,travel_class
102,Priya Reddy,Bangalore,Business


In [0]:

%sql
update passengers_delta
set travel_class = 'Premium Economy'
where passenger_id = 102;

num_affected_rows
1


In [0]:
%sql
insert into passengers_delta
values(
    111,
    'John',
    'New York',
    'First Class',
    'USA'
);

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
select passenger_id, passenger_name, travel_class from passengers_delta
where passenger_id = 102;

passenger_id,passenger_name,travel_class
102,Priya Reddy,Premium Economy


In [0]:
%sql
select * from passengers_delta;

passenger_id,passenger_name,city,travel_class,country
102,Priya Reddy,Bangalore,Premium Economy,India
101,Rahul Sharma,Hyderabad,Economy,India
103,Amit Kumar,Mumbai,Economy,India
105,Farhan Ali,Chennai,Economy,India
104,Sneha Patel,Hyderabad,Premium Economy,India
106,Neha Singh,Pune,Economy,India
107,Arjun Verma,Kochi,Business,India
131,John,New York,First Class,USA
131,John,New York,First Class,USA
111,John,New York,First Class,USA


In [0]:
%sql
SELECT passenger_id,
       passenger_name,
       city,
       travel_class
FROM default.passengers_delta
WHERE passenger_id = 106;

passenger_id,passenger_name,city,travel_class
106,Neha Singh,Pune,Economy


In [0]:
spark.read.format('delta').option('versionAsOf', 0).table('passengers_delta').filter('passenger_id = 102').show()

spark.read.format('delta').option('versionAsOf', 1).table('passengers_delta').filter('passenger_id = 102').show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore|    Business|  India|
+------------+--------------+---------+------------+-------+

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore| First Class|  India|
+------------+--------------+---------+------------+-------+



In [0]:
version1_df = spark.read.format('delta').option('versionAsOf', 0).table('passengers_delta')
version2_df = spark.read.format('delta').table('passengers_delta')

print("Version 0:", version1_df.count())
print("Version 1:", version2_df.count())

version1_df.filter('passenger_id = 102').show()
version2_df.filter('passenger_id = 102').show()

version1_df.filter('passenger_id = 104').show()
version2_df.filter('passenger_id = 104').show()

Version 0: 5
Version 1: 10
+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore|    Business|  India|
+------------+--------------+---------+------------+-------+

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         102|   Priya Reddy|Bangalore|Premium Economy|  India|
+------------+--------------+---------+---------------+-------+

+------------+--------------+-----+---------------+-------+
|passenger_id|passenger_name| city|   travel_class|country|
+------------+--------------+-----+---------------+-------+
|         104|   Sneha Patel|Delhi|Premium Economy|  India|
+------------+--------------+-----+---------------+-------+

+------------+--------------+---------+------

In [0]:
%sql
optimize passengers_delta;

path,metrics
,"List(1, 3, List(2126, 2126, 2126.0, 1, 2126), List(1716, 2072, 1834.6666666666667, 3, 5504), 0, null, null, 0, 1, 3, 0, true, 0, 0, 1781778204098, 1781778206280, 8, 1, null, List(0, 0), null, 5, 5, 516, 0, null, null)"


In [0]:
%sql
optimize passengers_delta
zorder by city;

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 2126), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781778227165, 1781778227685, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


In [0]:
%sql
delete from passengers_delta 
where passenger_id = 105;

num_affected_rows
1


In [0]:
%sql
vacuum passengers_delta;

path
""


In [0]:
%sql
describe history passengers_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
16,2026-06-18T10:24:40.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,VACUUM END,Map(status -> COMPLETED),null,List(3991796108455833),61ce80b3-3c9a-48d1-9f13-2417533d0a2e,0618-092705-x2bw3462-v2n,15,SnapshotIsolation,true,"Map(numDeletedFiles -> 0, numVacuumedDirectories -> 1)",null,Databricks-Runtime/18.2.x-photon-scala2.13
15,2026-06-18T10:24:39.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,VACUUM START,"Map(retentionCheckEnabled -> true, defaultRetentionMillis -> 604800000)",null,List(3991796108455833),61ce80b3-3c9a-48d1-9f13-2417533d0a2e,0618-092705-x2bw3462-v2n,14,SnapshotIsolation,true,"Map(numFilesToDelete -> 0, sizeOfDataToDelete -> 0)",null,Databricks-Runtime/18.2.x-photon-scala2.13
14,2026-06-18T10:24:04.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(passenger_id#21142L = 105)""])",null,List(3991796108455833),bad5dddb-8724-4b3c-924e-1b2904016ff9,0618-092705-x2bw3462-v2n,13,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1674, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1311, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 362)",null,Databricks-Runtime/18.2.x-photon-scala2.13
13,2026-06-18T10:23:26.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3991796108455833),9c3d6b95-1921-4aed-8448-2a341ea997af,0618-092705-x2bw3462-v2n,12,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 5504, p25FileSize -> 2126, numDeletionVectorsRemoved -> 0, minFileSize -> 2126, numAddedFiles -> 1, maxFileSize -> 2126, p75FileSize -> 2126, p50FileSize -> 2126, numAddedBytes -> 2126)",null,Databricks-Runtime/18.2.x-photon-scala2.13
12,2026-06-18T10:18:17.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3991796108455833),a8d84506-96b3-4c68-835e-cb6cdfa0e8e0,0618-092705-x2bw3462-v2n,11,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1716)",null,Databricks-Runtime/18.2.x-photon-scala2.13
11,2026-06-18T10:17:58.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3991796108455833),3199adfc-92b3-4407-85e3-b43ecada4e59,0618-092705-x2bw3462-v2n,10,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1716)",null,Databricks-Runtime/18.2.x-photon-scala2.13
10,2026-06-18T10:17:39.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3991796108455833),7bf393ba-5ad7-44af-ae4c-837f5ff3d185,0618-092705-x2bw3462-v2n,9,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 5570, p25FileSize -> 2072, numDeletionVectorsRemoved -> 1, minFileSize -> 2072, numAddedFiles -> 1, maxFileSize -> 2072, p75FileSize -> 2072, p50FileSize -> 2072, numAddedBytes -> 2072)",null,Databricks-Runtime/18.2.x-photon-scala2.13
9,2026-06-18T10:17:37.000Z,141685270976780,azuser7212_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(passenger_id#19323L = 102)""])",null,List(3991796108455833),7bf393ba-5ad7-44af-ae4c-837f5ff3d185,0618-092705-x2bw3462-v2n,8,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFile